# 04 · Polychromatic electric-field knots

**A self-contained, Colab-ready validation and exploration of the 2021 Physical Review Research paper.**

**Reference:** M. F. Ferrer-Garcia, A. D'Errico, H. Larocque, A. Sit and E. Karimi, *Polychromatic electric field knots*, Physical Review Research **3**, 033226 (2021). [Journal article](https://doi.org/10.1103/PhysRevResearch.3.033226) · [Published paper, including Appendices A–B](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.3.033226).

### Learning objectives

1. Understand the distinction between spatial polarization Möbius strips and locally time-knotted electric-field trajectories.
2. Implement the three vector-vortex modes of published Eqs. (4)–(5).
3. Focus every chromatic component with its **own wavenumber**, using the Richards–Wolf five-integral method.
4. Reconstruct the three-dimensional trajectory traced by the *real* instantaneous electric-field tip.
5. Investigate frequency ratios, phases, amplitudes, numerical convergence, and candidate knot regions.
6. Optionally compute a knot determinant with pyknotid, with appropriate numerical caveats.

**Reproduction level (physics-validation stage):** The physical construction and the paper's principal frequency/amplitude examples are implemented. Exact numerical datasets and all computational parameters underlying published knot histograms were not available, so plotted knot percentages are **not** claimed as exact replications. This notebook prioritizes scientific transparency and editable parameters.

**Runtime:** CPU is sufficient. Run sequentially or change the settings cell and then rerun the solver and visualization cells.

**Scientific review:** [validation findings, evidence and implementation sequence](04_Physics_Validation_Review.md). Knot mapping is outside the current development scope.


## 1. Theory: a knot traced in time, at one spatial point

The optical electric field is a sum of three monochromatic focused fields,

$$
\mathbf e(\mathbf r,t)=
\sum_{j=1}^{3}\operatorname{Re}\left[
a_j\mathbf E_j(\mathbf r;\omega_j)e^{-i\omega_jt}\right].
$$

The **curve** is the path of this real vector in electric-field component space, while the observation point $\mathbf r$ remains fixed. It is **not** a spatial curve in $(x,y,z)$ and is distinct from the director ribbons in Tutorial 03.

If $\omega_j=m_j\omega_0$ with integer, coprime harmonics $m_j$, let $\tau=\omega_0t$. The trajectory closes after $\tau\in[0,2\pi]$.

The incident vector vortex of published Eq. (4) is

$$
\mathbf V_{\ell,\delta}(\theta,\phi)=
\frac{h_{|\ell|}(\theta)}{\sqrt2}
\left[e^{i\ell\phi}\mathbf e_R+
e^{i\delta}e^{-i\ell\phi}\mathbf e_L\right],
$$

where $\mathbf e_R=(1,-i)^T/\sqrt2$, $\mathbf e_L=(1,+i)^T/\sqrt2$ use the explicit $e^{-i\omega t}$ convention. The Jones convention fixes any naming ambiguity.

For published Eq. (5), the three modes are $(\ell,\delta)=(1,0),(-1,0),(-1,\pi)$. The paper's example uses relative **incident** amplitudes $(1,2,2)$ and either frequency ratios $(1,3,2)$ or $(1,3/2,5/2)$. The second is equivalently represented by coprime integer harmonics $(2,3,5)$.

### Independent vectorial focusing of each colour

For each chromatic field separately we use

$$
\mathbf E_j(\mathbf r)=
-\frac{ik_jf}{2\pi}
\int_0^\alpha\int_0^{2\pi}
\sqrt{\cos\theta}\,
\mathbf R(\theta,\phi)\,\mathbf E_{{\rm in},j}(\theta,\phi)
\,e^{ik_j\mathbf s\cdot\mathbf r}
\sin\theta\,d\phi\,d\theta,
$$

with $k_j=2\pi n m_j/\lambda_{\rm ref}$ and $\alpha=\arcsin(\mathrm{NA}/n)$. The pupil envelope is
$h_{|\ell|}(\theta)=q^{|\ell|}e^{-(q/\mathrm{fill})^2}$, $q=\sin\theta/\sin\alpha$.

We omit a *common chosen lens reference phase* in the solver; real dispersive lens phases can be added as chromatic phases in the settings. The five Bessel orders after exact azimuthal integration are $\ell,\ell\pm1,\ell\pm2$. The longitudinal integrals include the additional $\sin\theta$ from the lens rotation. See Tutorial 01 for the derivation.

In [ ]:
%pip -q install numpy scipy matplotlib plotly ipywidgets
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.integrate import simpson
from scipy.special import jv
from functools import lru_cache, reduce
from math import gcd
from IPython.display import display, clear_output
import ipywidgets as widgets
print("NumPy", np.__version__)

## 2. Parameters to experiment with

Change this cell and rerun the cells below it. Lengths are expressed in the wavelength $\lambda_{\rm ref}$ corresponding to the **common base frequency** $\omega_0$. In the $(2,3,5)$ case, the longest optical wavelength is $\lambda_{\rm ref}/2$.

The original article's exact computational pupil fill and all plotting settings are not specified in the core text; the defaults here are clearly identified as **illustrative**. The amplitudes and frequencies reproduce the published choices.

The published **Figure 4** contains the knot maps and histograms; Figure 3 contains
the four monochromatic focused vector-vortex fields. We use the final journal
numbering (vector vortex: Eq. (4); three-colour superposition: Eq. (5)).
The paper specifies **NA = 0.95 in air** and equal pupil waists. Its radial
LG waist relative to the pupil aperture and the numerical convergence tolerances
are not specified in the published text. `FILL=1` remains an assumption.

Our length unit remains the base-frequency vacuum wavelength `LAMBDA_REF`.
The paper's wavelength is that of the *lowest optical frequency*:
$\lambda_{\min\omega}=\lambda_{\rm ref}/\min(m_j)$ in air.
Thus a fixed window in base-wavelength units covers twice as many paper-wavelength
units for (2,3,5) as for (1,3,2). To compare equal physical-size windows in the
paper's units, divide `HALF_WIDTH` and `Z` by `min(harmonics)` at fixed `LAMBDA_REF`.
This conversion matters for all reported spatial knot fractions.


In [ ]:
# --- User controls ---
PRESET = "paper_132"          # "paper_132", "paper_235", or "custom"
CUSTOM_HARMONICS = (1, 3, 2)
NA, n_medium = 0.95, 1.0       # published NA in air; requires NA < n
LAMBDA_REF, F = 1.0, 1.0       # vacuum base wavelength and focal length (length units)
FILL = 1.0                    # LG pupil envelope filling parameter
Z = 0.0                       # axial position / lambda_ref
HALF_WIDTH, NXY = 1.0, 81     # focal-plane grid (odd NXY)
NTHETA, NRADIAL = 401, 401    # integration and interpolation samples
AMPLITUDES = np.array([1.0,2.0,2.0])  # published Eq. (5) example
PHASES = np.array([0.,0.,0.])          # additional chromatic phases (radians)

presets = {"paper_132":(1,3,2),"paper_235":(2,3,5)}
if PRESET not in (*presets, "custom"):
    raise ValueError("Unknown preset; select paper_132, paper_235 or custom")
raw_harmonics = np.asarray(presets.get(PRESET,CUSTOM_HARMONICS), dtype=float)
if (raw_harmonics.shape != (3,) or not np.isfinite(raw_harmonics).all()
        or np.any(raw_harmonics <= 0) or np.any(raw_harmonics != np.floor(raw_harmonics))):
    raise ValueError("Supply three positive integer harmonics; do not truncate ratios")
harmonics = tuple(raw_harmonics.astype(int))
if len(set(harmonics)) != 3:
    raise ValueError("This three-colour tutorial requires distinct frequencies")
assert len(harmonics)==3 and min(harmonics)>0 and reduce(gcd,harmonics)==1
assert LAMBDA_REF>0 and F>0 and HALF_WIDTH>0
assert 0<NA<n_medium and FILL>0 and NXY>=5 and NXY%2==1
assert NTHETA>10 and NRADIAL>10
assert len(AMPLITUDES)==len(PHASES)==3
weights = np.asarray(AMPLITUDES,complex)*np.exp(1j*np.asarray(PHASES,float))
aperture = np.arcsin(NA/n_medium)
x = np.linspace(-HALF_WIDTH,HALF_WIDTH,NXY); y=x.copy()
X,Y=np.meshgrid(x,y,indexing="xy")
rho, azimuth=np.hypot(X,Y),np.arctan2(Y,X)
radial_nodes=np.linspace(0,float(rho.max()),NRADIAL)
theta=np.linspace(0,aperture,NTHETA)
print("Harmonics:",harmonics," Vacuum wavelengths:",[LAMBDA_REF/m for m in harmonics])

## 3. The numerical solver

The code builds a focused Jones basis for charges $+1$ and $-1$ at each of the three frequencies. It integrates the polar angle numerically after performing the azimuthal integration analytically. The output is complex128 with component order $(E_x,E_y,E_z)$.

The chromatic basis fields must be recomputed when any optical or spatial parameter is changed. Rerunning this definition cell resets the cache.

In [ ]:
R_JONES=np.array([1,-1j],complex)/np.sqrt(2)
L_JONES=np.array([1,+1j],complex)/np.sqrt(2)

def interpolate_radial(values):
    rr=rho.ravel()
    return (np.interp(rr,radial_nodes,values.real)
            +1j*np.interp(rr,radial_nodes,values.imag)).reshape(rho.shape)

@lru_cache(None)
def focused_basis(ell,harmonic):
    """Return focused unit-x and unit-y incident vortex modes: (2,3,Ny,Nx)."""
    k=2*np.pi*n_medium*harmonic/LAMBDA_REF
    T=theta[None,:]; U=radial_nodes[:,None]
    st=np.sin(T); ct=np.cos(T); q=st/np.sin(aperture)
    g=np.sqrt(ct)*st*q**abs(ell)*np.exp(-(q/FILL)**2)*np.exp(1j*k*Z*ct)
    arg=k*U*st
    def integrate(a): return simpson(a,x=theta,axis=1)
    b1=integrate(g*(1+ct)*jv(ell,arg))
    b2=integrate(g*(1-ct)*jv(ell+2,arg))
    b3=integrate(g*(1-ct)*jv(ell-2,arg))
    b4=integrate(g*st*jv(ell+1,arg)) # longitudinal sin(theta)
    b5=integrate(g*st*jv(ell-1,arg))
    fac=-np.pi*(1j**(ell+1))*F/(LAMBDA_REF/(n_medium*harmonic))
    flong=np.pi*(1j**ell)*F/(LAMBDA_REF/(n_medium*harmonic))
    I1=fac*interpolate_radial(b1)*np.exp(1j*ell*azimuth)
    I2=fac*interpolate_radial(b2)*np.exp(1j*(ell+2)*azimuth)
    I3=fac*interpolate_radial(b3)*np.exp(1j*(ell-2)*azimuth)
    I4=flong*interpolate_radial(b4)*np.exp(1j*(ell+1)*azimuth)
    I5=flong*interpolate_radial(b5)*np.exp(1j*(ell-1)*azimuth)
    Fx=np.stack(((2*I1+I2+I3)/2,(I2-I3)/(2j),I5-I4))
    Fy=np.stack(((I2-I3)/(2j),(2*I1-I2-I3)/2,1j*(I4+I5)))
    return np.stack((Fx,Fy))

def vector_vortex(ell,delta,harmonic):
    """Focus published Eq. (4): opposite charges in circular Jones components."""
    Bp=focused_basis(ell,harmonic)
    Bm=focused_basis(-ell,harmonic)
    return (R_JONES[0]*Bp[0]+R_JONES[1]*Bp[1]
            +np.exp(1j*delta)*(L_JONES[0]*Bm[0]+L_JONES[1]*Bm[1]))/np.sqrt(2)

focused_basis.cache_clear()
modes=[(+1,0),(-1,0),(-1,np.pi)]
E_colors=np.stack([w*vector_vortex(ell,d,h)
    for (ell,d),h,w in zip(modes,harmonics,weights)],axis=0)
assert E_colors.shape==(3,3,NXY,NXY) and np.isfinite(E_colors).all()
print("E_colors shape:",E_colors.shape)

### 3.1. Independent check against published Appendix A

This check integrates Eq. (A1) directly, using the first two columns of Eq. (A2).
It uses **Gauss–Legendre polar quadrature and a periodic azimuthal sum**;
it contains no Bessel functions and no radial interpolation. In contrast, the
production solver uses Simpson quadrature, analytic azimuthal integration and
linear radial interpolation. We compare *complex components*, without adjusting
their phase, amplitude or sign to fit the reference.

Write $c=\cos\theta$, $s=\sin\theta$. The unapodized transport is
$$R_{xy}=\begin{pmatrix}
c\cos^2\phi+\sin^2\phi &(c-1)\sin\phi\cos\phi\\
(c-1)\sin\phi\cos\phi &c\sin^2\phi+\cos^2\phi\\
-s\cos\phi&-s\sin\phi
\end{pmatrix}.$$
Apodization $\sqrt{c}$ and the measure $s\,d\theta d\phi$ are each included
exactly once. Each transported pupil vector is perpendicular to
$\hat{\mathbf s}=(s\cos\phi,s\sin\phi,c)$.
Using $\int e^{i\ell\phi}e^{iu\cos(\phi-\varphi)}d\phi
=2\pi i^\ell J_\ell(u)e^{i\ell\varphi}$ gives orders
$\ell,\ell\pm2$ transversely and $\ell\pm1$ longitudinally. The latter
carry the additional $s$ from the last row of $R$.

The published Eq. (1) uses $e^{-i\omega t}$, while Appendix A displays
$e^{+i\omega t}$. Here the spatial phasor integral is explicitly Eq. (A1)
with its displayed spatial signs and $-ikf/(2\pi)$ prefactor, reconstructed
with Eq. (1)'s negative time sign throughout. Reversing **every** temporal sign
reverses traversal of the same closed trajectory. Conjugating only selected
colours or dropping their intrinsic mode phases can instead change the curve.
This convention does not include an unspecified chromatic lens phase.

The envelope used here is a common-waist $p=0$ LG radial factor with
$q=r/(f\sin\alpha)$ and $w/(f\sin\alpha)=\mathrm{FILL}$, up to a common
constant for the $|\ell|=1$ modes. The frequency-dependent focusing prefactor
must be retained: normalizing each focal colour separately changes incident
amplitude ratios. Extension to unequal $|\ell|$ requires specifying their pupil
power normalization separately. General refractive-index settings are an
ideal homogeneous-medium extension; the publication benchmark is in air.


In [ ]:
def direct_rw_basis(ell,harmonic,points,npolar=80,nphi=192):
    """Independent A1/A2 reference. points: (N,3), physical length units.

    Returns (N, incident x/y basis, output x/y/z component).
    """
    from numpy.polynomial.legendre import leggauss
    nodes,gw=leggauss(npolar)
    th=(nodes+1)*aperture/2
    phi=2*np.pi*np.arange(nphi)/nphi
    st=np.sin(th)[:,None]; ct=np.cos(th)[:,None]
    cp=np.cos(phi)[None,:]; sp=np.sin(phi)[None,:]
    R=np.empty((npolar,nphi,3,2))
    R[:,:,0,0]=ct*cp**2+sp**2
    R[:,:,0,1]=(ct-1)*sp*cp
    R[:,:,1,0]=R[:,:,0,1]
    R[:,:,1,1]=ct*sp**2+cp**2
    R[:,:,2,0]=-st*cp
    R[:,:,2,1]=-st*sp
    direction=np.stack(np.broadcast_arrays(st*cp,st*sp,ct),axis=-1)
    assert np.max(abs(np.einsum("tpc,tpcb->tpb",direction,R)))<1e-13
    q=st/np.sin(aperture)
    pupil=q**abs(ell)*np.exp(-(q/FILL)**2)*np.exp(1j*ell*phi)[None,:]
    measure=gw[:,None]*(aperture/2)*(2*np.pi/nphi)*st*np.sqrt(ct)
    k=2*np.pi*n_medium*harmonic/LAMBDA_REF
    result=[]
    for point in np.asarray(points):
        phase=np.exp(1j*k*np.einsum("tpc,c->tp",direction,point))
        result.append((-1j*k*F/(2*np.pi)*
                       np.einsum("tp,tpcb->bc",measure*pupil*phase,R)))
    return np.asarray(result)

def verify_focus():
    # On-axis, generic off-axis, near-edge, and another azimuth; not only symmetry axes.
    indices=[(NXY//2,NXY//2),(NXY//2+2,NXY//2+1),
             (NXY//4,3*NXY//4),(NXY-2,NXY//3)]
    points=np.array([[x[i],y[j],Z] for j,i in indices])
    rows=[]
    for h in harmonics:
        for ell in (-1,1):
            ref=direct_rw_basis(ell,h,points)
            refined=direct_rw_basis(ell,h,points,npolar=120,nphi=256)
            candidate=np.stack([focused_basis(ell,h)[:,:,j,i] for j,i in indices])
            scale=max(float(np.max(abs(refined))),1e-30)
            component_errors=np.max(abs(candidate-refined),axis=(0,1))/scale
            oracle_error=float(np.max(abs(ref-refined))/scale)
            rows.append(dict(harmonic=h,ell=ell,
                ex=float(component_errors[0]),ey=float(component_errors[1]),
                ez=float(component_errors[2]),oracle_refinement=oracle_error))
    return rows

verification=verify_focus()
for row in verification: print(row)
print("Errors are absolute complex errors / largest reference component per case.")
assert max(row['oracle_refinement'] for row in verification)<1e-8
assert max(max(row[c] for c in ('ex','ey','ez')) for row in verification)<5e-4

# Analytic input-mode checks independent of the focusing integrals.
phi_test=np.linspace(0,2*np.pi,31,endpoint=False)
radial=(np.exp(1j*phi_test)[:,None]*R_JONES+
        np.exp(-1j*phi_test)[:,None]*L_JONES)/np.sqrt(2)
azimuthal=(np.exp(1j*phi_test)[:,None]*R_JONES-
           np.exp(-1j*phi_test)[:,None]*L_JONES)/np.sqrt(2)
assert np.allclose(radial,np.column_stack((np.cos(phi_test),np.sin(phi_test))))
assert np.allclose(azimuthal,-1j*np.column_stack((-np.sin(phi_test),np.cos(phi_test))))
print("Radial and azimuthal inputs, including intrinsic phase, verified.")


### 3.2. Separate diffraction convergence from topology convergence

The next table varies Simpson polar quadrature and radial interpolation separately,
holding the other resolution high. Errors use the independently refined direct
integral as reference; complex cancellation and near-zero components preclude
meaningful pointwise relative errors there. Repeat at nonzero `Z` and both presets.
The grid, time sampling and strand clearance still need their own convergence
study before any knot map can be reported.


In [ ]:
def focus_convergence():
    """Refine theta and radial interpolation separately; restore notebook state."""
    global theta,radial_nodes
    saved_theta,saved_radial=theta,radial_nodes
    indices=[(NXY//4,3*NXY//4),(NXY//2+2,NXY//2+1)]
    points=np.array([[x[i],y[j],Z] for j,i in indices])
    h=max(harmonics)  # most rapidly varying pupil phase/Bessel argument
    ref=direct_rw_basis(1,h,points,npolar=120,nphi=256)
    scale=max(float(np.max(abs(ref))),1e-30)
    rows=[]
    try:
        for axis in ('theta','radial'):
            for samples in (101,201,401):
                theta=np.linspace(0,aperture,samples if axis=='theta' else 801)
                # Probe radii are exact nodes in the theta sweep: no interpolation floor.
                radial_nodes=(np.linspace(0,float(rho.max()),samples) if axis=='radial'
                    else np.unique([0.,float(rho.max()),*[rho[j,i] for j,i in indices]]))
                focused_basis.cache_clear()
                actual=np.stack([focused_basis(1,h)[:,:,j,i] for j,i in indices])
                rows.append(dict(axis=axis,samples=samples,
                    relative_complex_error=float(np.max(abs(actual-ref))/scale)))
    finally:
        theta,radial_nodes=saved_theta,saved_radial
        focused_basis.cache_clear()
    return rows

convergence=focus_convergence()
for row in convergence: print(row)
print("These probe-point errors are not a global error bound or a topology guarantee.")


## 4. Inspect the chromatic fields

The summed background intensity shown here is a time-averaged squared-field proxy $\sum_j\|\mathbf E_j\|^2$, without cross-frequency terms. **A bright region does not prove that it contains knots.** We must inspect the *instantaneous* three-dimensional trajectories.

In [ ]:
I_modes=np.sum(abs(E_colors)**2,axis=1)
I_total=I_modes.sum(axis=0)
fig,axes=plt.subplots(1,4,figsize=(15,3.4),constrained_layout=True)
for j,ax in enumerate(axes):
    data=I_total if j==3 else I_modes[j]
    im=ax.imshow(data/max(I_total.max(),1e-30),origin="lower",
       extent=(x[0],x[-1],y[0],y[-1]),vmin=0,vmax=1,cmap="viridis")
    ax.set_title("All colors" if j==3 else f"Mode {j+1} (harmonic {harmonics[j]})")
    ax.set_xlabel("x / lambda_ref");ax.set_ylabel("y / lambda_ref");ax.set_aspect("equal")
fig.colorbar(im,ax=axes,shrink=.75,label="I / maximum total I")
plt.show()
for j in range(3):
    frac=np.sum(abs(E_colors[j,2])**2)/max(np.sum(I_modes[j]),1e-30)
    print(f"Longitudinal share of mode {j+1}: {frac:.4f}")

## 5. The field-tip trajectory and interactive explorer

The local time curve at a point $(x_i,y_j,z)$ is

$$
\mathbf C(\tau)=
\sum_{m=1}^3\operatorname{Re}\left[
\mathbf E_m(x_i,y_j,z)e^{-ih_m\tau}\right],
\qquad 0\le\tau\le 2\pi,
$$

where $h_m$ are the selected integer harmonics. The path is plotted in **electric-field component space** $(e_x,e_y,e_z)$, not in physical laboratory position space. The plot's coordinate normalization is graphical only; it preserves topology.

Use the index sliders to select a different focal point without recalculating diffraction.

In [ ]:
NT=700
tau=np.linspace(0,2*np.pi,NT+1)
temporal=np.exp(-1j*np.outer(np.array(harmonics),tau))

def local_curve(j,i):
    phasors=E_colors[:,:,j,i]              # color, component
    return np.real(np.einsum("mc,mt->tc",phasors,temporal))

def show_trajectory(j,i):
    curve=local_curve(j,i)
    scale=max(float(np.max(np.ptp(curve,axis=0))),1e-15)
    p=curve/scale
    fig=go.Figure()
    fig.add_trace(go.Scatter3d(x=p[:,0],y=p[:,1],z=p[:,2],mode="lines",
       line=dict(color=tau,colorscale="Turbo",width=7),name="tip trajectory"))
    fig.add_trace(go.Scatter3d(x=[p[0,0]],y=[p[0,1]],z=[p[0,2]],
       mode="markers",marker=dict(color="black",size=5),name="start/end"))
    fig.update_layout(title=f"Instantaneous E at (x,y)=({x[i]:.3f},{y[j]:.3f}) lambda_ref",
        scene=dict(aspectmode="data",xaxis_title="e_x (normalized)",
                   yaxis_title="e_y (normalized)",zaxis_title="e_z (normalized)"),
        height=620,margin=dict(l=0,r=0,b=0,t=45))
    fig.show()
    return curve

default_i=min(NXY-1,NXY//2+NXY//8)
default_j=min(NXY-1,NXY//2+NXY//8)
curve=show_trajectory(default_j,default_i)

In [ ]:
ix=widgets.IntSlider(value=default_i,min=0,max=NXY-1,step=1,
                     description="x pixel",continuous_update=False)
iy=widgets.IntSlider(value=default_j,min=0,max=NXY-1,step=1,
                     description="y pixel",continuous_update=False)
output=widgets.Output()
def redraw(change=None):
    with output:
        clear_output(wait=True)
        show_trajectory(iy.value,ix.value)
ix.observe(redraw,names="value");iy.observe(redraw,names="value")
display(widgets.VBox([ix,iy,output]))
redraw()

### 5.1. An analytical control: Figure 1 of the article

Before examining the tightly focused fields, reproduce the analytic **example curves** in Figure 1. The planar two-frequency case is

$$
\mathbf C_{\rm plane}(\tau)=
(\cos\tau+\sin2\tau,\;-\sin\tau+\cos2\tau,\;0),
$$

and the three-dimensional three-frequency example is

$$
\mathbf C_{\rm 3D}(\tau)=
(\sin3\tau,\;\cos\tau-2\cos2\tau,\;\sin\tau+2\sin2\tau).
$$

These curves are provided directly by the article. They validate the time-domain curve construction independently of the optical focusing implementation. The article's three-dimensional example is not the same as asserting every focal-plane trajectory is a nontrivial knot.

In [ ]:
t = np.linspace(0,2*np.pi,1601)
planar = np.column_stack((np.cos(t)+np.sin(2*t),
                         -np.sin(t)+np.cos(2*t),np.zeros_like(t)))
example_3d = np.column_stack((np.sin(3*t),
                 np.cos(t)-2*np.cos(2*t),np.sin(t)+2*np.sin(2*t)))
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=example_3d[:,0],y=example_3d[:,1],z=example_3d[:,2],
             mode="lines",line=dict(width=7,color=t,colorscale="Turbo"),
             name="Paper Fig. 1(b)"))
fig.update_layout(height=560,scene=dict(aspectmode="data",
    xaxis_title="E_x",yaxis_title="E_y",zaxis_title="E_z"),
    title="Analytical three-frequency trajectory from paper Fig. 1(b)")
fig.show()
fig2,ax=plt.subplots(figsize=(5,5))
ax.plot(planar[:,0],planar[:,1]);ax.set_aspect("equal");ax.set_xlabel("E_x")
ax.set_ylabel("E_y");ax.set_title("Planar two-frequency example, Fig. 1(a)")
plt.show()

## 6. Is it really a knot?

A three-dimensional closed path can look knotted while remaining an **unknot**. It can also be close to a self-intersection, where its numerical topology is unstable. The following preliminary diagnostics measure closure, sample steps and sampled vertex separation. They do not certify that the curve is embedded. Then the optional pyknotid package can calculate a knot determinant (the published paper cites pyknotid).

A reliably computed determinant larger than one establishes nontrivial knotting, but determinant one does **not** establish an unknot and the determinant does not uniquely identify every knot type. A full topological classification requires additional invariants and convergence checks.

In [ ]:
def curve_diagnostics(curve,skip_fraction=0.02):
    """Exhaustive vertex diagnostic, NOT a segment-intersection or topology test.

    Fixed time-fraction exclusion is consistent when temporal sampling changes.
    Segments can cross between vertices even when this distance is positive.
    """
    from scipy.spatial.distance import cdist
    p=np.asarray(curve,float)
    if p.ndim != 2 or p.shape[1] != 3 or len(p)<8 or not np.isfinite(p).all():
        raise ValueError("Expected at least eight finite 3D samples including closure")
    closure=np.linalg.norm(p[0]-p[-1])
    p=p[:-1]  # caller supplies the independently evaluated period endpoint
    count=len(p)
    skip=max(1,int(np.ceil(skip_fraction*count)))
    if skip>=count//2:
        raise ValueError("Exclusion removes all nonlocal vertex pairs")
    edges=np.linalg.norm(np.roll(p,-1,axis=0)-p,axis=1)
    near=np.inf
    for start in range(0,count,128):
        ids=np.arange(start,min(start+128,count))
        cyclic=np.abs(ids[:,None]-np.arange(count)[None,:])
        cyclic=np.minimum(cyclic,count-cyclic)
        distances=cdist(p[ids],p)
        distances[cyclic<=skip]=np.inf
        near=min(near,float(distances.min()))
    return {"closure":float(closure),"length":float(edges.sum()),
            "mean_step":float(edges.mean()),"nearest_nonlocal_vertex":near,
            "vertex_separation_to_step":float(near/max(edges.mean(),1e-30))}
print("Preliminary sample-vertex diagnostics (not a topology certificate):")
print(curve_diagnostics(curve))


In [ ]:
# OPTIONAL knot determinant (external package may require environment fixes).
# Uncomment the next line if you want a topological invariant:
# %pip -q install pyknotid
try:
    from pyknotid.spacecurves import Knot
    kobj=Knot(curve[:-1],verbose=False)
    det=kobj.determinant()
    print("Knot determinant:",det)
    print("Only classify a well-resolved, embedded, closed curve.")
except ImportError:
    print("pyknotid is optional. Install it by uncommenting the line above.")
except Exception as e:
    print("Topological identification unavailable:",type(e).__name__,str(e)[:250])

## 7. Explore the focal plane (not yet a knot-density classification)

The next two maps quantify (a) nonplanarity of the local trajectory through the ratio of the smallest and largest singular values of the centered sampled curve, and (b) a sampled-vertex distance diagnostic (which can miss segment crossings).

These **are not knot-type maps**. They help select promising points to investigate with full knot-invariant calculations without certifying that a trajectory is resolved or embedded. Reproducing the paper's Figure 4 knot statistics requires independent invariants on a large, well-resolved sample, numerical self-crossing screening, matching physical parameters and reporting error rates.

In [ ]:
STRIDE=5
ixs=np.arange(0,NXY,STRIDE);iys=np.arange(0,NXY,STRIDE)
nonplanar=np.empty((len(iys),len(ixs)))
separation=np.empty_like(nonplanar)
for jj,j in enumerate(iys):
    for ii,i in enumerate(ixs):
        c=local_curve(j,i)
        sv=np.linalg.svd(c[:-1]-c[:-1].mean(axis=0),compute_uv=False)
        nonplanar[jj,ii]=sv[-1]/max(sv[0],1e-30)
        separation[jj,ii]=curve_diagnostics(c)["vertex_separation_to_step"]
fig,axs=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
extent=(x[ixs[0]],x[ixs[-1]],y[iys[0]],y[iys[-1]])
for ax,array,label in zip(axs,[nonplanar,separation],
             ["Nonplanarity, NOT knot type","Vertex separation / step (not crossing test)"]):
    im=ax.imshow(array,origin="lower",extent=extent,cmap="viridis")
    ax.set_title(label);ax.set_xlabel("x / lambda_ref");ax.set_ylabel("y / lambda_ref")
    fig.colorbar(im,ax=ax)
plt.show()

## 8. Save reproducible fields and explore the physics

The NPZ export contains **all three complex chromatic focused fields**, not just a plotted snapshot. It can be loaded for different visualization or topology algorithms without repeating the diffraction integrals.

**Suggested experiments:** swap the two published harmonic presets; vary only one relative phase; suppress the first (radial) mode and inspect how longitudinal content changes; vary the aperture; move away from the focal plane by changing $z$; refine polar-angle quadrature and radial interpolation independently; and compare truly classified knot-density maps with Figure 4 of the article.

The source model excludes chromatic objective dispersion, material interfaces, polarization-dependent aberrations and all measurement noise. Relative chromatic phase is an experimental degree of freedom, not a cosmetic setting.

**Cite:** Ferrer-Garcia et al., Phys. Rev. Research 3, 033226 (2021), DOI 10.1103/PhysRevResearch.3.033226.

In [ ]:
from pathlib import Path
target=Path("polychromatic_electric_knots_fields.npz")
np.savez_compressed(target,E_colors=E_colors,x=x,y=y,z=Z,
    harmonics=np.array(harmonics),amplitudes=AMPLITUDES,phases=PHASES,
    NA=NA,n_medium=n_medium,lambda_ref=LAMBDA_REF,focal_length=F,
    fill=FILL,ntheta=NTHETA,nradial=NRADIAL,
    schema_version="tutorial04-validation-v1", time_convention="exp(-i*m*tau)",
    pupil_envelope="q**abs(ell)*exp(-(q/fill)**2)",
    lambda_lowest_optical=LAMBDA_REF/min(harmonics),
    length_unit="same physical units as lambda_ref", nt=NT,
    normalization="incident amplitudes; no per-colour focal normalization",
    ell=np.array([q[0] for q in modes]),delta=np.array([q[1] for q in modes]))
print("Saved",target.resolve())
try:
    from google.colab import files
    files.download(str(target))
except ImportError:
    print("Outside Colab: file is available in the working directory.")